# S3DF Pipeline Development Notebook

This notebook replicates the 5-stage optimization pipeline used by S3DF jobs (`single_track_optimization.py`).

**5-Stage Pipeline:**
1. Stage 0: Energy estimation via scan at origin
2. Stage 1: Hierarchical grid search for position + t0
3. Stage 2: Hierarchical cone direction search
4. Stage 3: Energy scan optimization
5. Stage 4: Adam optimizer refinement

**Configuration:** Uses `nrays_config` files (config_0 through config_8) with parameterized selection.

## Cell 1: Environment Setup and Imports

In [ ]:
import sys
sys.path.append('..')

# Standard imports
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from tqdm import tqdm
import json
import os
import time
import pickle
import glob
import uproot
from pathlib import Path
from jax import jit, value_and_grad
import optax
import subprocess

# LUCiD imports
from lucid.geometry import generate_detector
from lucid.generate import read_photon_data_from_photonsim
from lucid.simulation import setup_event_simulator
from lucid.utils import load_range_params, check_track_endpoint_in_detector
from lucid.detector_params import ParticleParams, load_detector_params

# Optimization imports
from lucid.optimization.grid_search import (
    load_optimization_config, 
    get_detector_bounds, 
    hierarchical_position_grid_search
)
from lucid.optimization.utils.functions import (
    hierarchical_direction_search_cone, 
    energy_scan_optimization,
    cartesian_to_spherical, 
    spherical_to_cartesian, 
    performance_summary,
    estimate_muon_energy_from_photon_count
)
from lucid.losses import origin_time_loss, counts_loss, cone_time_loss
from lucid.optimization.run import load_config

PHYSICS_CONFIG = '../config/SK_physics_config.json'

print("Environment setup complete")

## Cell 2: Configuration Selection (Parameterized)

Change `CONFIG_INDEX` to switch between config_0 through config_8.

| Config | nphot |
|--------|-------|
| 0 | 5,000 |
| 1 | 10,000 |
| 2 | 25,000 |
| 3 | 50,000 |
| 4 | 100,000 |
| **5** | **150,000** |
| 6 | 200,000 |
| 7 | 300,000 |
| 8 | 500,000 |

In [ ]:
# =====================================================================
# CONFIG PARAMETERIZATION - Change this to switch configs (0-8)
# =====================================================================
CONFIG_INDEX = 5

config_dir = Path('../s3df_jobs/nrays_config')
config_path = config_dir / f'opt_config_{CONFIG_INDEX}.json'
script_path = config_dir / 'create_configs.py'

if not config_path.exists():
    print("Config file not found. Creating it...")
    subprocess.run(
        [sys.executable, script_path.name],
        cwd=config_dir,
        check=True
    )
    print("Config file successfully created.")
else:
    print("Config file already exists.")

print(config_path)
adam_config = load_config(config_path)

# Load configuration
config = load_optimization_config(config_path)


# Add default values for parameters if not present (matching single_track_optimization.py)
if 'optimization_params' not in config:
    config['optimization_params'] = {}
config['optimization_params'].setdefault('damping_factor', 0.998)

if 'adam_optimizer' not in config:
    config['adam_optimizer'] = {}
config['adam_optimizer'].setdefault('learning_rate', 0.2)
config['adam_optimizer'].setdefault('b1', 0.9)
config['adam_optimizer'].setdefault('b2', 0.999)
config['adam_optimizer'].setdefault('eps', 1e-8)

# Display key parameters
print(f"Loaded Config {CONFIG_INDEX}")
print("=" * 50)
print(f"nphot:        {config['basic_config']['nphot']:,}")
print(f"n_events:     {config['basic_config']['n_events']}")
print(f"temperature:  {config['basic_config']['temperature']}")
print(f"k:            {config['basic_config']['k']}")
print(f"c_medium:     {config['basic_config']['c_medium']:.6f}")
print(f"data_dir:     {config['basic_config']['data_dir']}")
print(f"detector:     {config['basic_config']['default_json_filename']}")

## Cell 3: Detector and Simulator Setup

In [ ]:
config['basic_config']['data_dir']

In [ ]:
# Extract basic configuration
default_json_filename = '../config/SK_geom_config.json'
data_dir = '../data/water/muon/'
TEMPERATURE = 0.10
N_EVENTS = 20 #config['basic_config']['n_events']
K = config['basic_config']['k']
Nphot = 50_000 # config['basic_config']['nphot']
C_MEDIUM = config['basic_config']['c_medium']
qe = config['detector_params']['qe']

# Setup detector
print(f"Setting up detector from: {default_json_filename}")
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

# Get detector bounds
detector_bounds = get_detector_bounds(detector)
DETECTOR_R = detector_bounds.get('r', None)
DETECTOR_H = detector_bounds.get('H', None)

print(f"Detector type: {detector_bounds['type']}")
print(f"Detector R: {DETECTOR_R:.2f} m")
print(f"Detector H: {DETECTOR_H:.2f} m")
print(f"Number of sensors: {NUM_DETECTORS}")

# Load range parametrization for track endpoint validation
range_params = load_range_params('muon', 'water')
print(f"Loaded range parametrization: {range_params['description']}")

# Setup simulators
print("\nSetting up simulators...")
prediction_simulator = setup_event_simulator(
    default_json_filename, Nphot, TEMPERATURE, 
    max_sensors_per_cell=4, K=K, is_data=False,
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

data_simulator = setup_event_simulator(
    default_json_filename, Nphot, temperature=0.0, 
    K=K, is_data=True, is_calibration=False,
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

# Load detector parameters
detector_params = load_detector_params(PHYSICS_CONFIG)

print("Simulators ready!")
print(f"\nDetector params: scatter_length={detector_params.scatter_length}, wall_reflection_rate={detector_params.wall_reflection_rate}, "
      f"absorption_length={detector_params.absorption_length}, qe={detector_params.qe}")

In [ ]:
detector_params

## Cell 4: Event Generation Function

This matches `generate_event_data()` from `single_track_optimization.py`.

In [ ]:
def generate_event_data(event_idx, random_key, data_dir, data_simulator,
                       detector_bounds, fraction=0.9):
    """
    Generate a single event with random parameters within detector bounds.
    Matches single_track_optimization.py generate_event_data() function.
    
    Args:
        event_idx: Event index
        random_key: JAX random key
        data_dir: Directory containing .root files
        data_simulator: Data simulator function
        detector_bounds: Detector bounds dictionary
        fraction: Fraction of detector volume for position sampling
    """
    # Get all .root files in the directory
    root_files = sorted(glob.glob(os.path.join(data_dir, "*.root")))
    if not root_files:
        raise ValueError(f"No .root files found in directory: {data_dir}")

    # Randomly select a file using the random key
    file_select_key, random_key = jax.random.split(random_key)
    file_idx = jax.random.randint(file_select_key, shape=(), minval=0, maxval=len(root_files))
    data_file = root_files[int(file_idx)]

    # Get number of entries from the selected file
    with uproot.open(data_file) as file:
        tree = file['OpticalPhotons']
        n_entries = tree.num_entries

    entry_idx = event_idx % n_entries

    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

    # Process photon data
    photon_origins = photon_data['photon_origins']
    photon_directions = photon_data['photon_directions']
    photon_times = photon_data['photon_times']
    N = len(photon_origins)
    
    # Padding to 1_000_000 (hard coded in _simulation_core)
    padding_size = max(0, 1_000_000 - N)

    # Pad the origins array
    photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                        mode='constant', constant_values=0)

    # Pad the directions array with default unit vector [0,0,1]
    default_direction = jnp.array([0.0, 0.0, 1.0])
    padding_directions = jnp.tile(default_direction, (padding_size, 1))
    if padding_size > 0:
        photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
    else:
        photon_data['photon_directions'] = photon_directions

    # Pad the times array
    photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                          mode='constant', constant_values=0)
    photon_data['N'] = N

    key = random_key
    
    DETECTOR_R = detector_bounds['r']
    DETECTOR_H = detector_bounds['H']

    # Random position within detector bounds (cylindrical)
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=DETECTOR_R * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-DETECTOR_H/2 * fraction,
                               maxval=DETECTOR_H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])

    # Random direction on unit sphere
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

    true_energy = photon_data['energy']
    TRUE_T0 = jax.random.uniform(key, shape=(), minval=-15.0, maxval=15.0)

    true_track = ParticleParams.from_cartesian(
        energy=true_energy, position=true_position, direction=true_direction, t0=0.0
    )

    # Compute rotation to transform from original direction (0,0,1) to true_direction
    original_direction = jnp.array([0.0, 0.0, 1.0])
    true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)
    
    # Rotation axis = cross product of original and target directions
    rotation_axis = jnp.cross(original_direction, true_direction_norm)
    axis_norm = jnp.linalg.norm(rotation_axis)
    
    # Handle case where directions are parallel (axis_norm ~ 0)
    rotation_axis = jnp.where(
        axis_norm < 1e-6,
        jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
        rotation_axis / (axis_norm + 1e-8)
    )
    
    # Rotation angle = arccos of dot product
    rotation_angle = jnp.arccos(jnp.clip(
        jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
    ))
    
    # Set rotation parameters in photon_data
    photon_data['rotation_axis'] = rotation_axis
    photon_data['rotation_angle'] = rotation_angle
    photon_data['apply_rotation'] = jnp.array(True)
    
    # Set translation parameters to move from origin to true_position
    photon_data['apply_translation'] = jnp.array(True)
    photon_data['translation_vector'] = true_position
    
    key, _ = jax.random.split(key)
    true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))
    
    hit_counts, hit_times_raw = true_data
    hit_times = hit_times_raw + TRUE_T0

    return {
        'event_idx': event_idx,
        'entry_idx': entry_idx,
        'true_energy': float(true_energy),
        'true_position': np.array(true_position),
        'true_direction': np.array(true_direction),
        'TRUE_T0': float(TRUE_T0),
        'true_data': true_data,
        'hit_times': hit_times,
        'hit_counts': hit_counts,
        'photon_data': photon_data
    }

print("Event generation function defined.")

## Cell 5: Combined Loss Function Definition

In [ ]:
from lucid.losses import counts_loss, origin_time_loss, cone_time_loss
from lucid.optimization.run import run_complete_optimization_adam, load_config

def create_combined_loss_function(prediction_simulator):
    """Create combined loss function with specified parameters and return its gradient function"""

    @jit
    def combined_product_loss(params, hit_detector_positions, observed_times, observed_counts, 
                              true_data, key):
        """
        Combined loss function: product of vertex loss, counts loss, and time loss
        
        Args:
            params: [x, y, z, t0, theta, phi, energy] where theta and phi are spherical direction angles
        
        Returns:
            combined_loss, (vertex_loss_val, counts_loss_val, time_loss_val)
        """
        position = params[:3]
        t0 = params[3]
        theta = params[4]
        phi = params[5]
        energy = params[6]
        track = ParticleParams(
            energy=energy, position=position,
            theta=theta, phi=phi, t0=jnp.array(0.0)
        )
        log_w, flat_times, flat_indices, total_charge = prediction_simulator(track, key)
        simulated_counts = total_charge  # Already per-sensor (N_sensors,)
        # Compute per-sensor weighted average times
        weights = jnp.exp(log_w)
        weighted_times = jnp.zeros(NUM_DETECTORS).at[flat_indices].add(weights * flat_times)
        simulated_time = weighted_times / jnp.maximum(total_charge, 1e-10)
        
        vertex_loss_val = origin_time_loss(position, hit_detector_positions, observed_times,
                                           observed_counts, t0)
        counts_loss_val = counts_loss(observed_counts, simulated_counts)
        time_loss_val = cone_time_loss(observed_counts, simulated_time, observed_times, t0)
        
        combined_loss = jnp.sqrt((vertex_loss_val + 1e-6) * (counts_loss_val + 1e-6) * (time_loss_val + 1e-6))
        return combined_loss, (vertex_loss_val, counts_loss_val, time_loss_val)

    combined_grad_fn = jit(value_and_grad(combined_product_loss, has_aux=True))
    return combined_grad_fn, combined_product_loss

# Create combined gradient function
combined_grad_fn, combined_product_loss = create_combined_loss_function(prediction_simulator)

print("Combined loss function created.")

## Cell 6: Stage 0 - Energy Estimation at Origin

Initial energy estimation via scan at origin position with standard direction.

In [ ]:
def run_stage_0_energy_estimation(hit_detector_positions, observed_times, observed_counts,
                                   true_data, true_energy, verbosity=2):
    """
    Stage 0: Energy estimation via scan at origin position.
    Uses direction [1/sqrt(3), 1/sqrt(3), 1/sqrt(3)] (theta=arccos(1/sqrt(3)), phi=pi/4)
    """
    if verbosity >= 2:
        print("=" * 60)
        print("STAGE 0: Energy Estimation at Origin")
        print("=" * 60)
    
    # Scan energy at origin with standard direction
    stage0_results = energy_scan_optimization(
        prediction_simulator,
        position=jnp.array([0., 0., 0.]),
        theta=jnp.arccos(1/jnp.sqrt(3)),
        phi=jnp.pi/4.,
        initial_t0=0.,
        hit_detector_positions=hit_detector_positions,
        observed_times=observed_times,
        observed_charge=observed_counts,
        true_data=true_data,
        energy_guess=1000 + np.random.uniform(-50, 50),
        energy_delta=700,
        n_steps=10,
        verbosity=verbosity
    )
    
    if verbosity >= 2:
        print(f"\n  Energy guess: {stage0_results['best_energy']:.1f} MeV (true: {true_energy:.1f} MeV)")
        print(f"  Energy error: {abs(stage0_results['best_energy'] - true_energy):.1f} MeV")
    
    return stage0_results


def visualize_stage_0(stage0_results, true_energy):
    """Visualization: Energy vs Loss curve"""
    energies = [r['energy'] for r in stage0_results['scan_results']]
    losses = [r['loss'] for r in stage0_results['scan_results']]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(energies, losses, 'b-o', label='Energy Scan Loss', markersize=8)
    ax.axvline(true_energy, color='r', linestyle='--', linewidth=2, label=f'True E={true_energy:.0f} MeV')
    ax.axvline(stage0_results['best_energy'], color='g', linestyle='--', linewidth=2, 
               label=f'Best E={stage0_results["best_energy"]:.0f} MeV')
    ax.set_xlabel('Energy (MeV)', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.legend(fontsize=11)
    ax.set_title('Stage 0: Initial Energy Scan at Origin', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return fig

print("Stage 0 functions defined.")

## Cell 7: Stage 1 - Hierarchical Position + t0 Grid Search

In [ ]:
def run_stage_1_position_search(hit_detector_positions, observed_times, observed_counts,
                                 true_position, TRUE_T0, initial_t0, verbosity=2):
    """
    Stage 1: Hierarchical grid search for position + t0
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 1: Hierarchical Position + t0 Grid Search")
        print("=" * 60)
    
    pos_results = hierarchical_position_grid_search(
        hit_detector_positions, observed_times, observed_counts,
        true_position, TRUE_T0, initial_t0, detector_bounds,
        n_div=config['position_grid_search']['pos_n_div'],
        t0_n_div=config['position_grid_search']['t0_n_div'],
        levels=config['position_grid_search']['pos_levels'],
        fraction=config['position_grid_search']['pos_fraction'],
        t0_min=config['position_grid_search']['t0_min'],
        t0_max=config['position_grid_search']['t0_max'],
        min_L=config['position_grid_search']['pos_min_L'],
        verbosity=verbosity
    )
    
    if verbosity >= 2:
        print(f"\n  Best position: {pos_results['best_position']}")
        print(f"  Best t0: {pos_results['best_t0']:.3f} (true: {TRUE_T0:.3f})")
        print(f"  Position error: {pos_results['position_error']:.3f} m")
        print(f"  t0 error: {pos_results['t0_error']:.3f}")
    
    return pos_results


def visualize_stage_1(pos_results, true_position):
    """Visualization: 3D scatter of best positions per t0 level"""
    fig = go.Figure()
    
    # Add detector surface
    r, H = detector_bounds['r'], detector_bounds['H']
    x_cyl, y_cyl, z_cyl = create_cylinder_surface(r, H)
    fig.add_trace(go.Surface(
        x=x_cyl, y=y_cyl, z=z_cyl, 
        opacity=0.1, colorscale='Greys', showscale=False,
        name='Detector'
    ))
    
    # Plot best positions from t0 search (sample every 3rd)
    if 'all_t0_results' in pos_results:
        positions = []
        t0_values = []
        for t0_result in pos_results['all_t0_results'][::3]:
            if t0_result.get('best_position') is not None:
                positions.append(t0_result['best_position'])
                t0_values.append(t0_result['t0_value'])
        
        if positions:
            positions = np.array(positions)
            fig.add_trace(go.Scatter3d(
                x=positions[:, 0], y=positions[:, 1], z=positions[:, 2],
                mode='markers',
                marker=dict(size=4, color=t0_values, colorscale='Viridis', colorbar=dict(title='t0')),
                name='Grid Search Positions'
            ))
    
    # True position
    fig.add_trace(go.Scatter3d(
        x=[true_position[0]], y=[true_position[1]], z=[true_position[2]],
        mode='markers', 
        marker=dict(size=15, color='red', symbol='diamond'),
        name='True Position'
    ))
    
    # Best position
    if pos_results['best_position'] is not None:
        fig.add_trace(go.Scatter3d(
            x=[pos_results['best_position'][0]], 
            y=[pos_results['best_position'][1]], 
            z=[pos_results['best_position'][2]],
            mode='markers', 
            marker=dict(size=15, color='green', symbol='diamond'),
            name='Best Position'
        ))
    
    fig.update_layout(
        title='Stage 1: Position Grid Search',
        scene=dict(aspectmode='data'),
        width=900, height=700
    )
    fig.show()
    return fig

print("Stage 1 functions defined.")

## Cell 8: Stage 2 - Hierarchical Cone Direction Search

In [ ]:
def run_stage_2_direction_search(optimal_position, optimal_t0, energy_guess,
                                  hit_detector_positions, observed_times,
                                  observed_counts, true_data, true_direction, verbosity=2):
    """
    Stage 2: Hierarchical cone-based direction search
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 2: Hierarchical Cone Direction Search")
        print("=" * 60)
    
    cone_results = hierarchical_direction_search_cone(
        prediction_simulator, optimal_position, optimal_t0,
        hit_detector_positions, observed_times, observed_counts,
        true_data, energy_guess,
        levels=config['cone_direction_search']['cone_levels'],
        initial_div=config['cone_direction_search']['cone_initial_div'],
        max_angle_deg=config['cone_direction_search']['cone_max_angle_deg'],
        reduction=config['cone_direction_search']['cone_reduction'],
        verbosity=verbosity
    )
    
    # Calculate direction error
    best_dir = cone_results['best_direction']
    cos_angle = np.clip(np.dot(best_dir, true_direction), -1.0, 1.0)
    direction_error = np.degrees(np.arccos(cos_angle))
    cone_results['direction_error'] = direction_error
    
    if verbosity >= 2:
        print(f"\n  Best direction: {best_dir}")
        print(f"  True direction: {true_direction}")
        print(f"  Direction error: {direction_error:.2f} degrees")
    
    return cone_results


def visualize_stage_2(cone_results, true_direction):
    """Visualization: Unit sphere with sampled directions colored by loss"""
    fig = go.Figure()
    
    # Plot directions from each level
    colors = ['blue', 'green', 'orange', 'red']
    for i, level_data in enumerate(cone_results['search_path']):
        directions = level_data['directions']
        losses = [r['loss'] for r in level_data['direction_results']]
        
        fig.add_trace(go.Scatter3d(
            x=directions[:, 0], y=directions[:, 1], z=directions[:, 2],
            mode='markers',
            marker=dict(size=4, color=losses, colorscale='Viridis_r', 
                       showscale=(i == 0), colorbar=dict(title='Loss')),
            name=f"Level {level_data['level']}"
        ))
    
    # True direction
    fig.add_trace(go.Scatter3d(
        x=[true_direction[0]], y=[true_direction[1]], z=[true_direction[2]],
        mode='markers', 
        marker=dict(size=12, color='red', symbol='diamond'),
        name='True Direction'
    ))
    
    # Best direction
    best_dir = cone_results['best_direction']
    fig.add_trace(go.Scatter3d(
        x=[best_dir[0]], y=[best_dir[1]], z=[best_dir[2]],
        mode='markers', 
        marker=dict(size=12, color='lime', symbol='diamond'),
        name='Best Direction'
    ))
    
    fig.update_layout(
        title='Stage 2: Direction Cone Search (Unit Sphere)',
        scene=dict(aspectmode='cube'),
        width=800, height=700
    )
    fig.show()
    return fig

print("Stage 2 functions defined.")

## Cell 9: Stage 3 - Energy Scan Optimization

In [ ]:
def run_stage_3_energy_scan(optimal_position, best_theta, best_phi, optimal_t0,
                             energy_guess, hit_detector_positions, observed_times,
                             observed_counts, true_data, true_energy, verbosity=2):
    """
    Stage 3: Energy scan at optimal position/direction
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 3: Energy Scan Optimization")
        print("=" * 60)
    
    energy_results = energy_scan_optimization(
        prediction_simulator, optimal_position,
        best_theta, best_phi, optimal_t0,
        hit_detector_positions, observed_times, observed_counts,
        true_data, energy_guess,
        energy_delta=config['energy_optimization']['energy_delta'],
        n_steps=config['energy_optimization']['energy_scan_steps'],
        verbosity=verbosity
    )
    
    if verbosity >= 2:
        print(f"\n  Best energy: {energy_results['best_energy']:.1f} MeV (true: {true_energy:.1f} MeV)")
        print(f"  Energy error: {abs(energy_results['best_energy'] - true_energy):.1f} MeV")
        print(f"  Improvement from Stage 0: {energy_results['energy_improvement']:.1f} MeV")
    
    return energy_results


def visualize_stage_3(energy_results, stage0_results, true_energy):
    """Visualization: Energy vs Loss with Stage 0 comparison"""
    energies = [r['energy'] for r in energy_results['scan_results']]
    losses = [r['loss'] for r in energy_results['scan_results']]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(energies, losses, 'b-o', label='Stage 3 Energy Scan', markersize=6)
    ax.axvline(true_energy, color='r', linestyle='--', linewidth=2, label=f'True E={true_energy:.0f} MeV')
    ax.axvline(energy_results['best_energy'], color='g', linestyle='--', linewidth=2, 
               label=f'Stage 3 Best E={energy_results["best_energy"]:.0f} MeV')
    ax.axvline(stage0_results['best_energy'], color='orange', linestyle=':', linewidth=2,
               label=f'Stage 0 E={stage0_results["best_energy"]:.0f} MeV')
    ax.set_xlabel('Energy (MeV)', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.legend(fontsize=11)
    ax.set_title('Stage 3: Energy Scan at Optimal Position/Direction', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return fig

print("Stage 3 functions defined.")

## Cell 10: Stage 4 - Adam Optimizer Refinement

In [ ]:
def run_stage_4_adam_optimization(initial_params, hit_detector_positions, 
                                   observed_times, observed_counts, true_data,
                                   true_energy, true_position, true_direction, TRUE_T0,
                                   verbosity=2):
    """
    Stage 4: Adam optimizer refinement of all parameters.
    Matches single_track_optimization.py run_complete_optimization_adam() Stage 4.
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 4: Adam Optimizer Refinement")
        print("=" * 60)
    
    # Convert true direction to spherical
    true_theta, true_phi = cartesian_to_spherical(true_direction)
    
    # Adam parameters from config
    ADAM_LEARNING_RATE = config['adam_optimizer']['learning_rate']
    ADAM_B1 = config['adam_optimizer']['b1']
    ADAM_B2 = config['adam_optimizer']['b2']
    ADAM_EPS = config['adam_optimizer']['eps']
    MAX_ITERATIONS = config['gradient_descent']['max_iterations']
    damping_factor = config['optimization_params']['damping_factor']
    tolerance = 1e-6
    
    # Learning-rate scaling
    POS_LR_SCALE = config['learning_rates']['position_learning_rate']
    DIR_LR_SCALE = config['learning_rates']['direction_learning_rate']
    T0_LR_SCALE = config['learning_rates']['t0_learning_rate']
    ENE_LR_SCALE = config['learning_rates']['energy_learning_rate']
    
    update_scales = jnp.array([
        POS_LR_SCALE, POS_LR_SCALE, POS_LR_SCALE,  # x, y, z
        T0_LR_SCALE,                                # t0
        DIR_LR_SCALE, DIR_LR_SCALE,                 # theta, phi
        ENE_LR_SCALE                                # energy
    ])
    
    if verbosity >= 2:
        print(f"  Learning rate: {ADAM_LEARNING_RATE}")
        print(f"  Max iterations: {MAX_ITERATIONS}")
        print(f"  Update scales: pos={POS_LR_SCALE}, dir={DIR_LR_SCALE}, t0={T0_LR_SCALE}, E={ENE_LR_SCALE}")
    
    # Initialize optimizer
    optimizer = optax.adam(learning_rate=ADAM_LEARNING_RATE, b1=ADAM_B1, b2=ADAM_B2, eps=ADAM_EPS)
    opt_state = optimizer.init(initial_params)
    current_params = initial_params.copy()
    
    # History tracking
    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [],
        'vertex_losses': [],
        'counts_losses': [],
        'energy_losses': [],
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
    }
    
    opt_key = jax.random.PRNGKey(12345)
    current_damping_w = 5.0
    adam_start_time = time.time()

    MAX_ITERATIONS = 250
    for iteration in range(MAX_ITERATIONS):
        opt_key, _ = jax.random.split(opt_key)
        
        (combined_loss, (vertex_loss, counts_loss_val, energy_loss_val)), grad = combined_grad_fn(
            current_params, hit_detector_positions, observed_times, observed_counts,
            true_data, opt_key
        )
        
        # Handle NaN gradients
        if jnp.any(jnp.isnan(grad)):
            grad = jnp.nan_to_num(grad, nan=0.0)
        
        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break
        
        # Adam update with parameter-specific scaling
        updates, opt_state = optimizer.update(grad, opt_state, current_params)
        current_damping_w *= damping_factor
        scaled_updates = updates * update_scales * current_damping_w
        
        # Freeze energy for first 25 iterations
        if iteration < 25:
            scaled_updates = scaled_updates.at[-1].set(0.)
        
        current_params = optax.apply_updates(current_params, scaled_updates)
        
        # Apply constraints
        current_params = jnp.array([
            jnp.clip(current_params[0], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
            jnp.clip(current_params[1], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
            jnp.clip(current_params[2], -DETECTOR_H/2 * 0.95, DETECTOR_H/2 * 0.95),
            jnp.clip(current_params[3], -20.0, 20.0),
            current_params[4],
            current_params[5],
            jnp.clip(current_params[6], 300.0, 2000.0)
        ])
        
        # Calculate errors
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_direction = spherical_to_cartesian(current_params[4], current_params[5])
        current_energy = current_params[6]
        
        position_error = float(jnp.linalg.norm(current_position - true_position))
        t0_error = float(abs(current_t0 - TRUE_T0))
        energy_error = float(abs(current_energy - true_energy))
        cos_angle = np.clip(np.dot(np.array(current_direction), np.array(true_direction)), -1.0, 1.0)
        direction_error = float(np.degrees(np.arccos(cos_angle)))
        
        # Store history
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['vertex_losses'].append(float(vertex_loss))
        history['counts_losses'].append(float(counts_loss_val))
        history['energy_losses'].append(float(energy_loss_val))
        history['position_errors'].append(position_error)
        history['direction_errors'].append(direction_error)
        history['t0_errors'].append(t0_error)
        history['energy_errors'].append(energy_error)
        
        if ((iteration + 1) % 100 == 0 or iteration == 0):
            print(
                f"  Iter {iteration}: "
                f"loss={combined_loss:.6f} "
                f"cnt={counts_loss_val:.6f}), "
                f"pos_err={position_error:.3f}m, "
                f"dir_err={direction_error:.2f}deg, "
                f"t0_err={t0_error:.3f}, "
                f"E_err={energy_error:.1f}"
            )
    
    adam_end_time = time.time()
    
    # Final results
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)
    
    final_position_error = float(jnp.linalg.norm(final_position - true_position))
    final_t0_error = float(abs(final_t0 - TRUE_T0))
    final_energy_error = float(abs(final_energy - true_energy))
    cos_angle = np.clip(np.dot(np.array(final_direction), np.array(true_direction)), -1.0, 1.0)
    final_direction_error = float(np.degrees(np.arccos(cos_angle)))
    
    if verbosity >= 2:
        print(f"\n  Optimization completed in {adam_end_time - adam_start_time:.2f}s")
        print(f"  Final position error: {final_position_error:.3f} m")
        print(f"  Final direction error: {final_direction_error:.2f} deg")
        print(f"  Final t0 error: {final_t0_error:.3f}")
        print(f"  Final energy error: {final_energy_error:.1f} MeV")
    
    return {
        'initial_params': np.array(initial_params),
        'final_params': np.array(current_params),
        'final_position': np.array(final_position),
        'final_direction': np.array(final_direction),
        'final_theta': float(final_theta),
        'final_phi': float(final_phi),
        'final_t0': float(final_t0),
        'final_energy': float(final_energy),
        'final_position_error': final_position_error,
        'final_direction_error': final_direction_error,
        'final_t0_error': final_t0_error,
        'final_energy_error': final_energy_error,
        'final_combined_loss': history['combined_losses'][-1] if history['combined_losses'] else float('inf'),
        'final_vertex_loss': history['vertex_losses'][-1] if history['vertex_losses'] else float('inf'),
        'final_counts_loss': history['counts_losses'][-1] if history['counts_losses'] else float('inf'),
        'adam_optimization_time': adam_end_time - adam_start_time,
        'total_iterations': len(history['parameters']) - 1,
        'converged': grad_norm < tolerance,
        'history': history
    }


def visualize_stage_4(adam_results):
    """Visualization: Multi-panel loss and error curves"""
    history = adam_results['history']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Combined loss
    axes[0, 0].semilogy(history['combined_losses'])
    axes[0, 0].set_title('Combined Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Position error
    axes[0, 1].plot(history['position_errors'])
    axes[0, 1].set_title('Position Error (m)')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Direction error
    axes[0, 2].plot(history['direction_errors'])
    axes[0, 2].set_title('Direction Error (deg)')
    axes[0, 2].set_xlabel('Iteration')
    axes[0, 2].grid(True, alpha=0.3)
    
    # t0 error
    axes[1, 0].plot(history['t0_errors'])
    axes[1, 0].set_title('t0 Error')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Energy error
    axes[1, 1].plot(history['energy_errors'])
    axes[1, 1].set_title('Energy Error (MeV)')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Parameter trajectory (position components)
    params = np.array(history['parameters'])
    axes[1, 2].plot(params[:, 0], label='x')
    axes[1, 2].plot(params[:, 1], label='y')
    axes[1, 2].plot(params[:, 2], label='z')
    axes[1, 2].legend()
    axes[1, 2].set_title('Position Components (m)')
    axes[1, 2].set_xlabel('Iteration')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return fig

print("Stage 4 functions defined.")

## Cell 11: Main Processing Loop (10 Events)

Process N_EVENTS using the full 5-stage pipeline.

In [ ]:
# =====================================================================
# Main Processing Loop - 10 Events
# =====================================================================

# Storage for results
all_event_results = []

# Performance tracking
energy_guess_errors = []
grid_position_errors = []
grid_t0_errors = []
cone_direction_errors = []
energy_scan_improvements = []
final_position_errors = []
final_direction_errors = []
final_t0_errors = []
final_energy_errors = []
final_combined_losses = []
final_vertex_losses = []
final_counts_losses = []
convergence_rates = []

# Generate random keys
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

print(f"Processing {N_EVENTS} events...")
print("=" * 80)

for event_idx in range(N_EVENTS):
    event_start_time = time.time()
    
    print(f"\n{'='*80}")
    print(f"EVENT {event_idx}")
    print(f"{'='*80}")
    
    try:
        # Generate event with endpoint validation
        event_key = event_keys[event_idx]
        max_attempts = 10
        endpoint_valid = False
        
        for attempt in range(max_attempts):
            event_data = generate_event_data(
                event_idx, event_key, data_dir,
                data_simulator, detector_bounds, fraction=0.9
            )
            
            endpoint_valid = check_track_endpoint_in_detector(
                event_data['true_position'],
                event_data['true_direction'],
                event_data['true_energy'],
                range_params, detector_bounds, fraction=0.9
            )
            
            if endpoint_valid:
                break
            event_key, _ = jax.random.split(event_key)
        
        if not endpoint_valid:
            print(f"  ERROR: Could not generate valid event after {max_attempts} attempts")
            continue
        
        # Extract event data
        true_position = event_data['true_position']
        true_direction = event_data['true_direction']
        true_energy = event_data['true_energy']
        TRUE_T0 = event_data['TRUE_T0']
        true_data = event_data['true_data']
        
        # Get hit information
        hit_mask = event_data['hit_counts'] > -999
        hit_detector_positions = detector_points[hit_mask]
        observed_times = event_data['hit_times'][hit_mask]
        observed_counts = event_data['hit_counts'][hit_mask]
        
        print(f"  True position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}]")
        print(f"  True direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
        print(f"  True energy: {true_energy:.1f} MeV")
        print(f"  True t0: {TRUE_T0:.3f}")
        print(f"  N hits: {int(jnp.sum(hit_mask))}")
        
        # ===== STAGE 0: Energy estimation =====
        stage0_results = run_stage_0_energy_estimation(
            hit_detector_positions, observed_times, observed_counts,
            true_data, true_energy, verbosity=1
        )
        
        # ===== STAGE 1: Position + t0 grid search =====
        stage1_results = run_stage_1_position_search(
            hit_detector_positions, observed_times, observed_counts,
            true_position, TRUE_T0, initial_t0=0.0, verbosity=1
        )
        
        if stage1_results['best_position'] is None:
            print(f"  ERROR: Stage 1 failed")
            continue
        
        # ===== STAGE 2: Direction cone search =====
        stage2_results = run_stage_2_direction_search(
            stage1_results['best_position'], stage1_results['best_t0'],
            stage0_results['best_energy'], hit_detector_positions,
            observed_times, observed_counts, true_data, true_direction, verbosity=1
        )
        
        # ===== STAGE 3: Energy scan =====
        stage3_results = run_stage_3_energy_scan(
            stage1_results['best_position'], stage2_results['best_theta'],
            stage2_results['best_phi'], stage1_results['best_t0'],
            stage0_results['best_energy'], hit_detector_positions,
            observed_times, observed_counts, true_data, true_energy, verbosity=1
        )
        
        # ===== STAGE 4: Adam optimization =====
        initial_params = jnp.array([
            stage1_results['best_position'][0],
            stage1_results['best_position'][1],
            stage1_results['best_position'][2],
            TRUE_T0, #stage1_results['best_t0'],
            #stage1_results['best_t0'],
            stage2_results['best_theta'],
            stage2_results['best_phi'],
            stage3_results['best_energy']
        ])
        
        stage4_results = run_stage_4_adam_optimization(
            initial_params, hit_detector_positions, observed_times,
            observed_counts, true_data, true_energy,
            true_position, true_direction, TRUE_T0, verbosity=1
        )
        
        # Store results
        event_end_time = time.time()
        
        event_result = {
            'event_data': {
                'event_idx': event_data['event_idx'],
                'true_energy': event_data['true_energy'],
                'true_position': event_data['true_position'],
                'true_direction': event_data['true_direction'],
                'TRUE_T0': event_data['TRUE_T0'],
                'true_data': event_data['true_data'],
                'hit_detector_positions': hit_detector_positions,
                'observed_times': observed_times,
                'observed_counts': observed_counts,
            },
            'stage0': stage0_results,
            'stage1': stage1_results,
            'stage2': stage2_results,
            'stage3': stage3_results,
            'stage4': stage4_results,
            'optimization_results': stage4_results,  # For compatibility with visualization
            'total_event_time': event_end_time - event_start_time
        }
        all_event_results.append(event_result)
        
        # Track metrics
        energy_guess_errors.append(abs(stage0_results['best_energy'] - true_energy))
        grid_position_errors.append(stage1_results['position_error'])
        grid_t0_errors.append(stage1_results['t0_error'])
        cone_direction_errors.append(stage2_results['direction_error'])
        energy_scan_improvements.append(stage3_results['energy_improvement'])
        final_position_errors.append(stage4_results['final_position_error'])
        final_direction_errors.append(stage4_results['final_direction_error'])
        final_t0_errors.append(stage4_results['final_t0_error'])
        final_energy_errors.append(stage4_results['final_energy_error'])
        final_combined_losses.append(stage4_results['final_combined_loss'])
        final_vertex_losses.append(stage4_results['final_vertex_loss'])
        final_counts_losses.append(stage4_results['final_counts_loss'])
        convergence_rates.append(1.0 if stage4_results['converged'] else 0.0)
        
        print(f"\n  Event {event_idx} completed in {event_end_time - event_start_time:.2f}s")
        print(f"  Final: pos_err={stage4_results['final_position_error']:.3f}m, "
              f"dir_err={stage4_results['final_direction_error']:.2f}deg, "
              f"t0_err={stage4_results['final_t0_error']:.3f}, "
              f"E_err={stage4_results['final_energy_error']:.1f}MeV")
        
    except Exception as e:
        print(f"  ERROR processing event {event_idx}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"Completed processing {len(all_event_results)} events successfully")
print(f"{'='*80}")

In [ ]:
# Performance summary
if len(all_event_results) > 0:
    print("\n")
    performance_summary(
        energy_guess_errors,
        grid_position_errors,
        cone_direction_errors,
        energy_scan_improvements,
        final_position_errors,
        final_direction_errors,
        final_t0_errors,
        final_energy_errors,
        final_combined_losses,
        final_vertex_losses,
        final_counts_losses,
        [],  # final_energy_losses not tracked separately
        convergence_rates,
    )
    
    # t0 grid search statistics
    print("\n" + "=" * 80)
    print("4D Grid Search t0 Performance:")
    print("=" * 80)
    grid_t0_errors_arr = np.array(grid_t0_errors)
    print(f"Mean t0 error:   {np.mean(grid_t0_errors_arr):.4f}")
    print(f"Median t0 error: {np.median(grid_t0_errors_arr):.4f}")
    print(f"Std t0 error:    {np.std(grid_t0_errors_arr):.4f}")
    
    # Timing summary
    print("\n" + "=" * 80)
    print("TIMING SUMMARY")
    print("=" * 80)
    total_times = [e['total_event_time'] for e in all_event_results]
    adam_times = [e['stage4']['adam_optimization_time'] for e in all_event_results]
    print(f"Total event time - Mean: {np.mean(total_times):.2f}s, Median: {np.median(total_times):.2f}s")
    print(f"Adam time - Mean: {np.mean(adam_times):.2f}s, Median: {np.median(adam_times):.2f}s")
    print(f"Adam % of total: {100 * np.sum(adam_times) / np.sum(total_times):.1f}%")
else:
    print("No events processed successfully.")

## Cell 13: Interactive 3D Visualization for Selected Event

In [ ]:
# Select event to visualize
EVENT_TO_VISUALIZE = 0

if len(all_event_results) > EVENT_TO_VISUALIZE:
    event_result = all_event_results[EVENT_TO_VISUALIZE]
    
    # Get true charges and times for visualization
    true_charges = event_result['event_data']['true_data'][0]
    true_times = event_result['event_data']['true_data'][1]
    
    # Create figures directory
    figures_dir = 'figures/'
    os.makedirs(figures_dir, exist_ok=True)
    
    # Event 3D visualization
    print(f"Creating 3D event visualization for Event {EVENT_TO_VISUALIZE}...")
    fig_event = create_event_3D_visualization(
        EVENT_TO_VISUALIZE, all_event_results, detector_points,
        true_charges, true_times, detector_bounds, 
        color_by='charge', min_charge=2.5, 
        figures_dir=figures_dir, detector_name='SK'
    )
    fig_event.show()
else:
    print(f"Event {EVENT_TO_VISUALIZE} not available.")

## Cell 14: Visualize Single Event with All Stages

In [ ]:
# Visualize all stages for a single event
if len(all_event_results) > EVENT_TO_VISUALIZE:
    event = all_event_results[EVENT_TO_VISUALIZE]
    true_energy = event['event_data']['true_energy']
    true_position = event['event_data']['true_position']
    true_direction = event['event_data']['true_direction']
    
    print(f"\nVisualizing all stages for Event {EVENT_TO_VISUALIZE}")
    print("=" * 60)
    
    # Stage 0
    print("\nStage 0: Energy estimation")
    visualize_stage_0(event['stage0'], true_energy)
    
    # Stage 1
    print("\nStage 1: Position grid search")
    visualize_stage_1(event['stage1'], true_position)
    
    # Stage 2
    print("\nStage 2: Direction cone search")
    visualize_stage_2(event['stage2'], true_direction)
    
    # Stage 3
    print("\nStage 3: Energy scan")
    visualize_stage_3(event['stage3'], event['stage0'], true_energy)
    
    # Stage 4
    print("\nStage 4: Adam optimization")
    visualize_stage_4(event['stage4'])